In [0]:
raw_path='/mnt/bayerhackathon/'
for tbl in list:
    path=raw_path+tbl+'.csv'
    {tbl}_df = spark.read.format("csv").option("header", "true").load(path)

In [0]:

order_df= spark.read.format("csv").option("header", "true").load("/mnt/bayerhackathon/order.csv")
customer_behaviour_df=spark.read.format("csv").option("header", "true").load("/mnt/bayerhackathon/customer_behaviour.csv")
customer_SCD2_data_df=spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/bayerhackathon/customer_SCD2_data.csv")
customer_df=spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/bayerhackathon/customer.csv")
order_line_df=spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/bayerhackathon/order_line.csv")


In [0]:
for tbl in list:
    {tbl}_df.write.format("delta").mode("overwrite").saveAsTable("tbl_BRONZE_"+{tbl})

In [0]:
order_df.write.format("delta").mode("overwrite").saveAsTable("tbl_BRONZE_order")
customer_behaviour_df.write.format("delta").mode("overwrite").saveAsTable("tbl_BRONZE_customer_behaviour")
customer_df.write.format("delta").mode("overwrite").saveAsTable("tbl_BRONZE_customer")
order_line_df.write.format("delta").mode("overwrite").saveAsTable("tbl_BRONZE_order_line")

In [0]:
customer_clean_query=f"""select * from tbl_BRONZE_customer where phone is not null"""
clean_customer_df= spark.sql(customer_clean_query)

customer_behaviour_clean_query=f"""select cb.* from tbl_BRONZE_customer_behaviour cb left join tbl_BRONZE_customer c on cb.customer_id = c.customer_id where c.phone is not null"""
customer_behaviour_df= spark.sql(customer_behaviour_clean_query)

order_clean_query = f"""select o.* from tbl_BRONZE_order o inner join tbl_BRONZE_customer c on o.customer_id = c.customer_id where c.phone is not null"""
order_cleand_df=spark.sql(order_clean_query)

order_line_cleaned_df=order_line_df.join(order_cleand_df, order_line_df.order_id == order_cleand_df.order_id, how='leftanti')


In [0]:
for tbl in list:
    {tbl}_df.write.format("delta").mode("overwrite").saveAsTable("tbl_SILVER_"+{tbl})

In [0]:
order_cleand_df.write.format("delta").mode("overwrite").saveAsTable("tbl_SILVER_order")
customer_behaviour_df.write.format("delta").mode("overwrite").saveAsTable("tbl_SILVER_customer_behaviour")
clean_customer_df.write.format("delta").mode("overwrite").saveAsTable("tbl_SILVER_customer")
order_line_cleaned_df.write.format("delta").mode("overwrite").saveAsTable("tbl_SILVER_order_line")
